---
sidebar_label: TensorFeed
---

# TensorFeed

>[TensorFeed.ai](https://tensorfeed.ai) is an AI news and real-time data hub built for humans and AI agents. It tracks AI news from 12+ industry sources, live up/down status for major AI services, current per-token model pricing, and public benchmark scores.

The `langchain-tensorfeed` package exposes four tools an agent can call to ground answers in current data. Every endpoint used here is free and requires no API key.

## Overview

### Tools

| Tool | Description |
| --- | --- |
| `TensorFeedNewsTool` | Latest AI news headlines, filterable by category. |
| `TensorFeedStatusTool` | Live up/down status for Claude, ChatGPT, Gemini, Copilot, Perplexity, Mistral, HuggingFace, and more. |
| `TensorFeedPricingTool` | Per-token pricing for AI models across all major providers. |
| `TensorFeedBenchmarksTool` | Benchmark scores across MMLU, HumanEval, GPQA, MATH, SWE-Bench, and others. |

### Setup

No credentials are required. The TensorFeed public API is open and rate-limited per IP.

### Installation

Install the integration package from PyPI:

In [ ]:
%pip install --quiet --upgrade langchain-tensorfeed

## Instantiation

Each tool is a standalone `BaseTool` subclass. Construct them with no arguments to use the production TensorFeed API.

In [ ]:
from langchain_tensorfeed import (
    TensorFeedBenchmarksTool,
    TensorFeedNewsTool,
    TensorFeedPricingTool,
    TensorFeedStatusTool,
)

news_tool = TensorFeedNewsTool()
status_tool = TensorFeedStatusTool()
pricing_tool = TensorFeedPricingTool()
benchmarks_tool = TensorFeedBenchmarksTool()

## Invocation

### Get the latest AI news

In [ ]:
news_tool.invoke({"category": "anthropic", "limit": 5})

### Check whether a service is up

In [ ]:
status_tool.invoke({"service": "claude"})

Omit the `service` argument to get a summary of every tracked provider.

In [ ]:
status_tool.invoke({})

### Look up model pricing

Filter by provider, model substring, or both.

In [ ]:
pricing_tool.invoke({"provider": "openai"})

In [ ]:
pricing_tool.invoke({"model": "sonnet"})

### Pull benchmark scores

In [ ]:
benchmarks_tool.invoke({"benchmark": "MMLU"})

## Use within an agent

All four tools work with any LangChain agent. Here is a tool-calling agent that grounds its answers in TensorFeed data:

In [ ]:
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate

from langchain_tensorfeed import (
    TensorFeedBenchmarksTool,
    TensorFeedNewsTool,
    TensorFeedPricingTool,
    TensorFeedStatusTool,
)

tools = [
    TensorFeedNewsTool(),
    TensorFeedStatusTool(),
    TensorFeedPricingTool(),
    TensorFeedBenchmarksTool(),
]

llm = ChatAnthropic(model="claude-opus-4-7")

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an AI industry analyst. Use the TensorFeed tools to ground every answer in current data.",
        ),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
)

agent = create_tool_calling_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

executor.invoke(
    {"input": "Is Claude up right now, and how does its pricing compare to GPT-4o?"}
)

## API reference

- Source code: [github.com/RipperMercs/tensorfeed](https://github.com/RipperMercs/tensorfeed) under `sdk/langchain-python/`
- TensorFeed developer docs: [tensorfeed.ai/developers](https://tensorfeed.ai/developers)
- API metadata: [tensorfeed.ai/api/meta](https://tensorfeed.ai/api/meta)